# Notebook 1 — Phylogenetics, Orthology & Paralogy

## Objectives
- Understand **orthology** and **paralogy** from gene/species tree reconciliation
- Use **ETE3** to load, annotate, and visualize phylogenetic trees
- Query the **NCBI taxonomy** database programmatically
- Visualize alignments: **ETE3 SeqMotifFace** and **pyMSAviz**
- Build a multiple sequence alignment with **MAFFT**
- Trim alignments: **trimAl** and a custom gap-filter with **pandas**
- Compute basic **alignment statistics**

## Orthology and Paralogy

![Species/Gene tree](../figures/fig1_orthology.png "Species and Gene tree")

**Orthologs** arise from **speciation (S)** — they are the "same gene" in different species and typically retain the same function.

**Paralogs** arise from **gene duplication (D)** — they are related genes within a genome that may diverge in function (sub- or neo-functionalization).

Distinguishing orthologs from paralogs is fundamental in comparative genomics: orthologs are our best bet for transferring functional annotations across species, while paralogs tell us about gene family expansion and functional diversification.

---
## 1. Simple gene tree: Cytochrome c oxidase I (COX1)

COX1 is a mitochondrial gene widely used as a phylogenetic marker. The tree has sequences from three species — human, mouse, and fruit fly. We use [ETE3](http://etetoolkit.org/) for tree visualization inside the notebook.

In [ ]:
from ete3 import PhyloTree, TreeStyle, faces

# Load tree — taxid is the part before the dot in each leaf name
t = PhyloTree(open("../data/cox1.nw").read(),
              sp_naming_function=lambda node: node.name.split('.')[0])

# Root at midpoint
t.set_outgroup(t.get_midpoint_outgroup())
print(t)

In [ ]:
# Annotate with NCBI taxonomy (downloads DB on first run)
print("Annotating with NCBI taxonomy...")
t.annotate_ncbi_taxa()

print(t.get_ascii(attributes=['sci_name', 'name']))

In [ ]:
# Render inline — default style
t.render("%%inline")
# t.show()

In [ ]:
# Customize the tree style
ts = TreeStyle()
ts.show_leaf_name = False
ts.show_scale = False
ts.show_branch_length = True
ts.show_branch_support = True

t.render("%%inline", tree_style=ts)

In [ ]:
# Custom layout: control how each node looks
def cox1_layout(node):
    if node.is_leaf():
        node.img_style['size'] = 10
        node.img_style["shape"] = "sphere"
        node.img_style["fgcolor"] = "darkred"
        name = faces.TextFace(
            f"{node.name.split('.')[1]} — {node.sci_name}", fstyle="italic")
        name.margin_left = 5
        if node.taxid == 9606:
            name.fgcolor = "steelblue"
        faces.add_face_to_node(name, node, column=0)
    else:
        node.img_style['size'] = 6
        node.img_style["shape"] = "sphere"
        node.img_style["fgcolor"] = "green"

t.render("%%inline", tree_style=ts, layout=cox1_layout)

### Evolutionary events

ETE can infer whether internal nodes represent **speciation** or **duplication** events by comparing species overlap between descendant clades.

With only one gene per species, COX1 should show only speciation events.

In [ ]:
events = t.get_descendant_evol_events()

for node in t.traverse():
    if 'evoltype' in node.features:
        print(node.name, node.evoltype)

print(t.get_ascii(attributes=['evoltype', 'name']))

---
## 2. Gene family duplications: SLC23A (vitamin C transporters)

The SLC23A family has four paralogs in mouse, three in human, and one in *Drosophila*. We need to understand **duplication nodes** clearly before scaling up.

In [ ]:
# Load the SLC23A tree
t = PhyloTree(open("../data/slc23a.nw").read(),
              sp_naming_function=lambda node: node.name.split('.')[0])

# Midpoint rooting
t.set_outgroup(t.get_midpoint_outgroup())

# Annotate
t.annotate_ncbi_taxa()
print(t.get_ascii(attributes=["name", "sci_name"]))

Now root the tree using the *Drosophila melanogaster* sequence as outgroup — this is biologically more meaningful than midpoint rooting.

In [ ]:
t.set_outgroup("7227.Q9VH02")
print(t.get_ascii(attributes=["name", "sci_name"]))

In [ ]:
# Re-annotate after re-rooting and infer duplication/speciation events
t.annotate_ncbi_taxa()
events = t.get_descendant_evol_events()

n_dup = sum(1 for e in events if e.etype == "D")
n_spec = sum(1 for e in events if e.etype == "S")
print(f"Duplication events: {n_dup}")
print(f"Speciation events: {n_spec}")

# Load gene names
seqid2gene = {}
for line in open("../data/slc23a.seqid2gname.tab"):
    parts = line.strip().split("\t")
    if len(parts) >= 2:
        seqid2gene[parts[0]] = parts[1]

print("\nGene names:")
for leaf in t.get_leaves():
    gname = seqid2gene.get(leaf.name, "?")
    print(f"  {leaf.name:<25s} {leaf.sci_name:<25s} {gname}")

### ✏️ Exercise 1

**Task:** Render the SLC23A tree inline with:
1. **Duplication** nodes marked with red squares
2. Each species in a different color
3. Leaf labels showing both the gene name and the species

Hint: write a `layout` function that checks `node.evoltype` for internal nodes and uses `faces.TextFace` for leaves. Use `t.render("%%inline", tree_style=ts, layout=your_layout)`.

In [ ]:
# YOUR CODE

---
## 3. The NCBI Taxonomy database

ETE provides a local interface to the [NCBI Taxonomy](https://www.ncbi.nlm.nih.gov/taxonomy) — a single hierarchical classification of all organisms.

In [ ]:
from ete3 import NCBITaxa

ncbi = NCBITaxa()

# Translate taxids to species names
taxid2name = ncbi.get_taxid_translator([9606, 10090, 7227])
print(taxid2name)

In [ ]:
# Build a taxonomy tree for these species
tree = ncbi.get_topology([9606, 10090, 7227])
print(tree.get_ascii(attributes=["sci_name", "rank"]))

### ✏️ Exercise 2

Use `ncbi.get_lineage()` and `ncbi.get_taxid_translator()` to print the full taxonomic lineage for taxid 9739 (bottlenose dolphin). What order does it belong to?

Then build a taxonomy tree (`ncbi.get_topology()`) for human, mouse, dog (9615), cow (9913), dolphin (9739), and horseshoe bat (59479). Render it inline.

In [ ]:
# YOUR CODE HERE

---
## 4. Alignment visualization

Before working with large alignments, let's learn two tools for visualizing MSAs.

### 4.1 Tree + alignment with ETE3 SeqMotifFace

ETE3's `SeqMotifFace` can display aligned sequences next to tree leaves.
We use the SLC23A tree from above with its trimmed alignment.

In [ ]:
import os

DATA = os.path.join("..", "data")
FIGS = os.path.join("..", "figures")

# Read the SLC23A trimmed alignment
def read_fasta(path):
    """Read FASTA file → dict of {header: sequence}."""
    seqs = {}
    header = None
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                header = line[1:].split()[0]
                seqs[header] = ""
            elif header:
                seqs[header] += line
    return {h: s for h, s in seqs.items()}

slc23a_aln = read_fasta(os.path.join(DATA, "slc23a.mft.gt01.fa"))
print(f"SLC23A alignment: {len(slc23a_aln)} sequences × {len(next(iter(slc23a_aln.values())))} columns")

In [1]:
from ete3 import SeqMotifFace

# Re-load the SLC23A tree
t_slc23a = PhyloTree(open("../data/slc23a.nw").read(),
                     sp_naming_function=lambda node: node.name.split('.')[0])
t_slc23a.set_outgroup("7227.Q9VH02")
t_slc23a.annotate_ncbi_taxa()

def aln_layout(node):
    if node.is_leaf():
        # Gene name label
        gname = seqid2gene.get(node.name, node.name.split('.')[1])
        sci = getattr(node, 'sci_name', '')
        label = faces.TextFace(f" {gname} ({sci})", fsize=10)
        label.margin_left = 4
        faces.add_face_to_node(label, node, column=0)
        # Alignment
        seq = slc23a_aln.get(node.name, "")
        if seq:
            seqface = SeqMotifFace(seq, seq_format="seq", height=10, gap_format="line")
            seqface.margin_left = 10
            faces.add_face_to_node(seqface, node, column=1, aligned=True)

ts = TreeStyle()
ts.show_leaf_name = False
ts.show_scale = False
ts.tree_width = 120

t_slc23a.render("%%inline", tree_style=ts, layout=aln_layout)
# t_slc23a.show(tree_style=ts, layout=aln_layout)

NameError: name 'PhyloTree' is not defined

The alignment is drawn next to the tree — gaps appear as lines. You can
see which positions differ between closely vs distantly related sequences.

### 4.2 Standalone alignment visualization with pyMSAviz

[pyMSAviz](https://github.com/moshi4/pyMSAviz) renders MSA plots with
matplotlib — color schemes, consensus bars, markers. Great for publication
figures and quick inspection.

In [ ]:
from pymsaviz import MsaViz

SLC23A_ALN = os.path.join(DATA, "slc23a.mft.gt01.fa")

mv = MsaViz(SLC23A_ALN, format="fasta", wrap_length=80,
            show_consensus=True, show_count=True)
mv.plotfig()

In [ ]:
# Try a different color scheme
mv = MsaViz(SLC23A_ALN, format="fasta", color_scheme="Identity",
            wrap_length=80, show_consensus=True)
mv.plotfig()

### ✏️ Exercise 3

1. Try `color_scheme="Identity"` — what do the colors tell?
2. Add markers to highlight positions with low conservation:
   ```python
   mv = MsaViz(SLC23A_ALN, format="fasta", show_consensus=True)
   ident_list = mv._get_consensus_identity_list()
   low = [pos+1 for pos, ident in enumerate(ident_list) if ident < 50]
   mv.add_markers(low, color="red", marker="v")
   mv.plotfig()
   ```
3. What fraction of positions have > 80% identity?

In [ ]:
# YOUR CODE HERE

---
## 5. The SLC26 family

We have 297 canonical protein sequences from 30 mammalian species, selected by having both Pfam domains PF01740 (Sulphate_transp) and PF00916 (STAS). The FASTA was prepared from UniProt reference proteomes (see `scripts/uniprot_phylo.py`). Headers use the format `>taxid.accession`.

In [ ]:
import subprocess
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter

sns.set_theme(style="whitegrid")

FASTA = os.path.join(DATA, "selection2_clustalo.fa")

# Look at the data
seqs = read_fasta(FASTA)
taxids = set(h.split(".")[0] for h in seqs)

print(f"Sequences: {len(seqs)}")
print(f"Species (unique taxids): {len(taxids)}")

### ✏️ Exercise 4

Compute the **sequence length distribution** and plot a histogram with seaborn.
Are all sequences similar in length, or are there outliers?

Hint: `sns.histplot(lengths)` or `sns.displot(lengths)`.

In [ ]:
# YOUR CODE HERE

---
## 6. Multiple sequence alignment with MAFFT

These SLC26 proteins share only ~20–40% identity across subfamilies, making alignment non-trivial. We use [MAFFT](https://mafft.cbrc.jp/alignment/software/) (`--auto` selects the best strategy based on dataset size - FAST).

Other popular aligners: [ClustalOmega](http://www.clustal.org/omega/), [MUSCLE](https://drive5.com/muscle5/).

In [ ]:
ALN = os.path.join(DATA, "slc26.mafft.fa")

if not os.path.exists(ALN):
    print("Running MAFFT (--auto)...")
    with open(ALN, "w") as f:
        subprocess.run(["mafft", "--auto", "--thread", "-1", FASTA],
                       stdout=f, stderr=subprocess.PIPE, check=True)
    print("Done.")
else:
    print(f"Cached: {ALN}")

aln = read_fasta(ALN)
aln_len = len(next(iter(aln.values())))
print(f"Alignment: {len(aln)} sequences × {aln_len} columns")

### 6.1 Alignment statistics with pandas

In [ ]:
# Convert alignment to DataFrame: rows = sequences, columns = positions
df_aln = pd.DataFrame({sid: list(seq) for sid, seq in aln.items()}).T

print(f"Alignment as DataFrame: {df_aln.shape}")

# Gap fraction per column
gap_frac = (df_aln == "-").mean()


In [ ]:
# Check df_aln == "-"


In [ ]:
print(f"Columns with 0 gaps:    {(gap_frac == 0).sum()}")
print(f"Columns with >50% gaps: {(gap_frac > 0.5).sum()}")
print(f"Columns with >90% gaps: {(gap_frac > 0.9).sum()}")
print(f"Mean gap fraction:      {gap_frac.mean():.3f}")

In [ ]:
#sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(range(len(gap_frac)), gap_frac.values, width=1, color="steelblue", alpha=0.7)
ax.axhline(0.9, color="red", ls="--", lw=0.8, label="90% gaps (trimAl -gt 0.1)")
ax.set(xlabel="Alignment column", ylabel="Gap fraction",
       title="Per-column gap fraction — SLC26 MAFFT alignment")
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

---
## 7. Alignment trimming

Columns dominated by gaps add noise (and time) to tree inference. We remove them before building the tree.

### 7.1 trimAl

[trimAl](http://trimal.cgenomics.org/) is a widely used tool. The flag `-gt 0.1` keeps only columns where at least 10% of sequences have a non-gap character (i.e., removes columns with >90% gaps).

In [ ]:
gt = 0.1
gt_str = "".join(str(gt).split("."))

TRIM = os.path.join(DATA, f"slc26.mafft.gt{gt_str}.fa")

if not os.path.exists(TRIM):
    subprocess.run(["trimal", "-in", ALN, "-out", TRIM, "-gt", str(gt), "-fasta"],
                   check=True, capture_output=True)

aln_trim = read_fasta(TRIM)
trim_len = len(next(iter(aln_trim.values())))
print(f"Before trim: {aln_len} columns")
print(f"After trim:  {trim_len} columns  ({aln_len - trim_len} removed)")

### ✏️ Exercise 5a

Try different trimAl settings: `-gt 0.3` (stricter) and `-gappyout` (automatic heuristic). How many columns does each retain? Which would you use for tree building? Vizualize the trimmed alignments.

In [ ]:
# YOUR CODE HERE

### ✏️ Exercise 5b — Write your own gap-trimming function with pandas

Write a function `trim_by_gaps(aln_dict, threshold)` that takes a dict of
`{seqid: aligned_sequence}` and a gap threshold (0 to 1), and returns a new
dict keeping only columns where the fraction of non-gap characters ≥ threshold.

This is essentially what `trimal -gt` does.

In [ ]:
def trim_by_gaps(aln_dict, threshold=0.1):
    """Keep columns where fraction of non-gap chars >= threshold."""
    # YOUR CODE HERE
    pass

# Test it:
# my_trim = trim_by_gaps(aln, threshold=0.1)
# my_trim_len = len(next(iter(my_trim.values())))
# print(f"Custom trim: {my_trim_len} columns")
# print(f"trimAl:      {trim_len} columns")

---
## Summary

- **Orthologs** (speciation) vs **paralogs** (duplication) — the key distinction in comparative genomics
- **ETE3**: load trees, annotate with NCBI taxonomy, infer evolutionary events, render inline
- **NCBI Taxonomy**: query taxids, build taxonomy trees
- **Alignment visualization**: ETE3 `SeqMotifFace` (tree + alignment) and **pyMSAviz** (standalone)
- **MAFFT**: fast multiple sequence alignment (alternatives: ClustalOmega, MUSCLE)
- **Trimming**: remove gappy columns before tree building (trimAl or pandas)

We have our SLC26 alignment trimmed and ready. In **Notebook 2** we build the tree, explore it interactively with **ETE4**, identify the prestin subfamily, and compare phylogenetic methods.